# Banking Intent Detection Chatbot

This notebook demonstrates the inference pipeline for a fine-tuned DistilBERT model on the BANKING77 dataset.

Features:
- Load trained model
- Predict banking intent
- Confidence score
- Top-3 predicted intents
- Retrieve similar customer queries
- Banking response generation
- Interactive chatbot

## PIPELINE

In [1]:
# Importing Libraries

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import (
DistilBertTokenizerFast,
DistilBertForSequenceClassification
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device : ", device)

Device :  cpu


In [3]:
#Load Dataset

train_df = pd.read_csv("../Data/train.csv")
train_df

,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival
...,...,...
9998,You provide support in what countries?,country_support
9999,What countries are you supporting?,country_support
10000,What countries are getting support?,country_support
10001,Are cards available in the EU?,country_support


In [4]:
#Label Encoder

encoder = LabelEncoder()
encoder.fit(train_df["category"])
num_classes = len(encoder.classes_)
print("Total Intents : ", num_classes)

Total Intents :  77


In [5]:
# Loading Tokenizer

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "models/saved_model"
)

In [6]:
#Load Model

model = DistilBertForSequenceClassification.from_pretrained(
    "models/saved_model"
)
model.to(device)
model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [7]:
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.distilbert(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.cpu().numpy()

In [8]:
train_embeddings = []

for i, text in enumerate(train_df["text"]):

    emb = get_embedding(text)

    train_embeddings.append(emb.squeeze())

    if (i + 1) % 100 == 0:
        print(f"Processed {i+1}/{len(train_df)}")

Processed 100/10003
Processed 200/10003
Processed 300/10003
Processed 400/10003
Processed 500/10003
Processed 600/10003
Processed 700/10003
Processed 800/10003
Processed 900/10003
Processed 1000/10003
Processed 1100/10003
Processed 1200/10003
Processed 1300/10003
Processed 1400/10003
Processed 1500/10003
Processed 1600/10003
Processed 1700/10003
Processed 1800/10003
Processed 1900/10003
Processed 2000/10003
Processed 2100/10003
Processed 2200/10003
Processed 2300/10003
Processed 2400/10003
Processed 2500/10003
Processed 2600/10003
Processed 2700/10003
Processed 2800/10003
Processed 2900/10003
Processed 3000/10003
Processed 3100/10003
Processed 3200/10003
Processed 3300/10003
Processed 3400/10003
Processed 3500/10003
Processed 3600/10003
Processed 3700/10003
Processed 3800/10003
Processed 3900/10003
Processed 4000/10003
Processed 4100/10003
Processed 4200/10003
Processed 4300/10003
Processed 4400/10003
Processed 4500/10003
Processed 4600/10003
Processed 4700/10003
Processed 4800/10003
P

In [9]:
np.save(
    "models/train_embeddings.npy",
    train_embeddings
)

In [10]:
train_embeddings = np.load(
    "models/train_embeddings.npy"
)

In [11]:
def retrieve_examples(query, top_k=3):

    query_embedding = get_embedding(query)

    similarities = cosine_similarity(
        query_embedding,
        train_embeddings
    )[0]

    indices = similarities.argsort()[-top_k:][::-1]

    return train_df.iloc[indices]

In [12]:
def predict(query):

    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

    probs = F.softmax(outputs.logits,dim=1)

    confidence,prediction = torch.max(probs,dim=1)

    top3 = torch.topk(probs,3)

    return (
        prediction.item(),
        confidence.item(),
        top3.indices.squeeze().cpu().numpy(),
        top3.values.squeeze().cpu().numpy()
    )

In [13]:
responses = {

    "card_arrival":"Your card is currently being processed and will be delivered shortly.",

    "cash_withdrawal":"You can withdraw cash using your physical card from any supported ATM.",

    "declined_card_payment":"Your payment was declined. Please verify your balance.",

    "change_pin":"You can change your PIN from the banking app.",

    "forgotten_passcode":"Use the 'Forgot Passcode' option to reset your passcode."

}

In [14]:
def chatbot(query):

    pred_id, confidence, top3_ids, top3_scores = predict(query)

    intent = encoder.inverse_transform([pred_id])[0]

    print("="*80)
    print("BANKING INTENT CHATBOT")
    print("="*80)

    print("\nUser Query:")
    print(query)

    print("\nPredicted Intent:")
    print(intent)

    print(f"\nConfidence: {confidence*100:.2f}%")

    print("\nTop 3 Predictions:")

    for idx, score in zip(top3_ids, top3_scores):

        print(
            f"{encoder.inverse_transform([idx])[0]} : {score*100:.2f}%"
        )

    print("\nBank Response:")

    print(
        responses.get(
            intent,
            "Thank you for contacting the bank. We will assist you shortly."
        )
    )

    print("\nSimilar Customer Queries:")

    similar = retrieve_examples(query)

    for _, row in similar.iterrows():

        print(f"• {row['text']}")

    print("="*80)

In [17]:
chatbot("I want to change my ATM PIN")

BANKING INTENT CHATBOT

User Query:
I want to change my ATM PIN

Predicted Intent:
change_pin

Confidence: 67.57%

Top 3 Predictions:
change_pin : 67.57%
get_physical_card : 5.47%
pin_blocked : 2.29%

Bank Response:
You can change your PIN from the banking app.

Similar Customer Queries:
• Which atms allow me to change my pin?
• What ATMs will allow me to change my PIN?
• Which ATM's am i able to change my PIN?


In [18]:
chatbot("I have forgotten my passcode")

BANKING INTENT CHATBOT

User Query:
I have forgotten my passcode

Predicted Intent:
passcode_forgotten

Confidence: 76.15%

Top 3 Predictions:
passcode_forgotten : 76.15%
verify_top_up : 1.44%
edit_personal_details : 1.10%

Bank Response:
Thank you for contacting the bank. We will assist you shortly.

Similar Customer Queries:
• I have forgotten my passcode
• I happened to forget my passcode
• I think I forgot my passcode


In [19]:
chatbot("My card payment was declined.")

BANKING INTENT CHATBOT

User Query:
My card payment was declined.

Predicted Intent:
declined_card_payment

Confidence: 78.33%

Top 3 Predictions:
declined_card_payment : 78.33%
reverted_card_payment? : 4.32%
card_not_working : 1.80%

Bank Response:
Your payment was declined. Please verify your balance.

Similar Customer Queries:
• Tell me why my card payment was declined.
• Tell me why my card payment has been declined.
• My card payment has been declined, why?


In [20]:
chatbot("What can I do if my card hasen't arrived ?")

BANKING INTENT CHATBOT

User Query:
What can I do if my card hasen't arrived ?

Predicted Intent:
card_arrival

Confidence: 67.17%

Top 3 Predictions:
card_arrival : 67.17%
card_delivery_estimate : 8.24%
lost_or_stolen_card : 2.07%

Bank Response:
Your card is currently being processed and will be delivered shortly.

Similar Customer Queries:
• My card hasn't arrived.
• My card was supposed to arrive, but hasn't?
• What's the reason my new card hasn't come?


In [21]:
chatbot("What is the reason for not card arrival ?")

BANKING INTENT CHATBOT

User Query:
What is the reason for not card arrival ?

Predicted Intent:
card_arrival

Confidence: 37.75%

Top 3 Predictions:
card_arrival : 37.75%
card_delivery_estimate : 8.84%
transfer_not_received_by_recipient : 6.29%

Bank Response:
Your card is currently being processed and will be delivered shortly.

Similar Customer Queries:
• Is there a reason my new card hasn't arrived?
• What's the reason my new card hasn't come?
• My card was supposed to arrive, but hasn't?
